# FMP Provider Adapter (Slice 5B)

Slice 5A put a provider-neutral boundary under OwnerLens. Slice 5B plugs a second provider into it:

```
SEC Company Facts ─→ sec_adapter ─┐
                                  ├─→ CanonicalFinancialHistory ─→ existing OwnerLens calculations
FMP statements   ─→ fmp_adapter ─┘
```

FMP (Financial Modeling Prep) supplies **provider-normalized reported line items**. OwnerLens maps only the line items its canonical vocabulary needs and still computes FCF, FCF/share, margins, ROIC, growth, capital-allocation ratios, and every classification itself. FMP's precomputed `freeCashFlow`, ratios, and growth are never used as OwnerLens truth.

This notebook walks one company from FMP response → canonical facts → OwnerLens economics, then runs the six validation companies on the free FMP tier. It needs `OWNER_LENS_FMP_API_KEY` in the environment or `.env`; the key is never printed.

In [1]:
from pathlib import Path

from owner_lens import (
    CANONICAL_METRICS,
    FmpClient,
    FmpError,
    MetricStatus,
    canonical_history_from_fmp,
    capital_efficiency_from_history,
    company_coverage_from_history,
    economic_value_summary_from_history,
    load_settings,
    owner_economics_from_history,
    require_fmp_api_key,
)
from owner_lens.fmp_adapter import FMP_FIELD_MAP

env_file = next((str(p) for p in (Path('.env'), Path('../.env')) if p.exists()), None)
api_key = require_fmp_api_key(load_settings(env_file=env_file))  # FmpConfigurationError if missing
client = FmpClient(api_key)

adbe_statements = client.retrieve_annual_statements('ADBE', limit=5)  # free tier: limit <= 5
for url in adbe_statements.source_urls:
    print(url)  # retrieval context, API key redacted

https://financialmodelingprep.com/stable/income-statement?symbol=ADBE&period=annual&limit=5&apikey=<redacted>
https://financialmodelingprep.com/stable/balance-sheet-statement?symbol=ADBE&period=annual&limit=5&apikey=<redacted>
https://financialmodelingprep.com/stable/cash-flow-statement?symbol=ADBE&period=annual&limit=5&apikey=<redacted>


## 1. The FMP response

FMP returns one row per fiscal year per statement. Each row carries its own identity and filing context (`date`, `filingDate`, `reportedCurrency`), many standardized line items, and FMP's sign conventions: cash outflows are negative and unreported line items are filled with `0`.

In [2]:
latest_cf = adbe_statements.cash_flow[0]
context = {k: latest_cf[k] for k in ('symbol', 'date', 'fiscalYear', 'period', 'reportedCurrency', 'filingDate')}
print(context)
for field in ('netCashProvidedByOperatingActivities', 'investmentsInPropertyPlantAndEquipment',
              'capitalExpenditure', 'commonStockRepurchased', 'commonDividendsPaid', 'freeCashFlow'):
    print(f'{field:<40} {latest_cf[field]:>18,}')

{'symbol': 'ADBE', 'date': '2025-11-28', 'fiscalYear': '2025', 'period': 'FY', 'reportedCurrency': 'USD', 'filingDate': '2026-01-15'}
netCashProvidedByOperatingActivities         10,031,000,000
investmentsInPropertyPlantAndEquipment         -179,000,000
capitalExpenditure                             -179,000,000
commonStockRepurchased                      -11,281,000,000
commonDividendsPaid                                       0
freeCashFlow                                  9,852,000,000


## 2. FMP → canonical facts

`canonical_history_from_fmp` applies a fixed, documented field map. Some of its policies are FMP-specific:

* Outflows (capex, repurchases, dividends) become positive magnitudes, the canonical convention SEC already follows. A positive outflow is `INVALID`.
* FMP's as-reported line items are preferred over its standardized variants. In live ADBE FY2021, `capitalExpenditure` (330M) differs from `investmentsInPropertyPlantAndEquipment` (348M, matching the 10-K).
* Fields FMP does not supply (period start, form, accession) stay `None`; nothing is fabricated.
* All-zero dividends or short-term investments are `STRUCTURALLY_ABSENT`. A zero revenue, asset, or share count is treated as not supplied, never as a value.

In [3]:
adbe = canonical_history_from_fmp(adbe_statements)

print(f"{'canonical metric':<26} {'FMP field':<40} FY2025 value")
for spec in CANONICAL_METRICS:
    series = adbe.series_for(spec.name)
    latest = series.observations[0].value if series.observations else series.status.value
    print(f'{spec.name:<26} {FMP_FIELD_MAP[spec.name].field:<40} {latest:>,}' if isinstance(latest, int)
          else f'{spec.name:<26} {FMP_FIELD_MAP[spec.name].field:<40} {latest}')

fact = adbe.series_for('repurchases').observations[0]
print()
print('provenance:', {k: getattr(fact, k) for k in (
    'provider', 'provider_field', 'fiscal_year', 'period_start', 'period_end', 'form', 'filed', 'accession')})

canonical metric           FMP field                                FY2025 value
revenue                    revenue                                  23,769,000,000
operating_income           operatingIncome                          8,706,000,000
net_income                 netIncome                                7,130,000,000
operating_cash_flow        netCashProvidedByOperatingActivities     10,031,000,000
capital_expenditures       investmentsInPropertyPlantAndEquipment   179,000,000
diluted_shares             weightedAverageShsOutDil                 427,000,000
income_tax_expense         incomeTaxExpense                         1,604,000,000
pretax_income              incomeBeforeTax                          8,734,000,000
repurchases                commonStockRepurchased                   11,281,000,000
stock_based_compensation   stockBasedCompensation                   1,942,000,000
dividends_paid             commonDividendsPaid                      STRUCTURALLY_ABSENT
cash        

## 3. Existing OwnerLens calculations on FMP-backed history

The calculation functions are the same ones Slice 5A introduced. They receive a `CanonicalFinancialHistory` and cannot tell which provider produced it, except through provenance.

In [4]:
owner = owner_economics_from_history(adbe)
capital = {row.fiscal_year: row for row in capital_efficiency_from_history(adbe)}
print(f"{'FY':>4} {'FCF':>16} {'FCF/share':>10} {'FCF margin':>11} {'ROIC':>7}")
for row in owner:
    roic = capital[row.fiscal_year].roic
    print(f'{row.fiscal_year:>4} {row.free_cash_flow:>16,} {row.fcf_per_share:>10.2f} '
          f'{row.fcf_margin:>11.1%} {(f"{roic:.1%}" if roic is not None else "n/a"):>7}')

summary = economic_value_summary_from_history(adbe)
print()
print('ADBE overall (FMP-backed):', summary.overall_economic_value_classification.value)
print('FMP freeCashFlow field (not used):', f"{latest_cf['freeCashFlow']:,}")

  FY              FCF  FCF/share  FCF margin    ROIC
2025    9,852,000,000      23.07       41.4%   61.6%
2024    7,824,000,000      17.39       36.4%   44.8%
2023    6,942,000,000      15.12       35.8%   43.5%
2022    7,396,000,000      15.70       42.0%   38.3%
2021    6,882,000,000      14.31       43.6%     n/a

ADBE overall (FMP-backed): IMPROVING
FMP freeCashFlow field (not used): 9,852,000,000


With five free-tier fiscal years, the earliest year has no prior balance sheet, so its ROIC is honestly `n/a` rather than computed from a fabricated baseline.

## 4. Free-tier live validation: six companies

This step confirms only that FMP calls succeed, canonical histories are produced, and OwnerLens calculations run where data is available. Comparing SEC-backed and FMP-backed values is Slice 5C (reconciliation).

In [5]:
for ticker in ('ADBE', 'V', 'COST', 'MSFT', 'CRM', 'NOW'):
    try:
        statements = adbe_statements if ticker == 'ADBE' else client.retrieve_annual_statements(ticker, limit=5)
    except FmpError as exc:
        print(f'{ticker:<5} FMP {type(exc).__name__}: {str(exc)[:90]}')
        continue
    history = canonical_history_from_fmp(statements)
    gaps = {m: s.value for m, s in history.statuses().items() if s is not MetricStatus.AVAILABLE}
    coverage = company_coverage_from_history(history)
    layers = sum(r.state.value == 'AVAILABLE' for r in coverage.layers.values())
    overall = economic_value_summary_from_history(history).overall_economic_value_classification.value
    years = [f.fiscal_year for f in history.series_for('revenue').observations]
    print(f'{ticker:<5} FY{years[-1]}-FY{years[0]}  layers available {layers}/6  overall {overall:<20} gaps {gaps or "none"}')

client.close()

ADBE  FY2021-FY2025  layers available 6/6  overall IMPROVING            gaps {'dividends_paid': 'STRUCTURALLY_ABSENT'}


V     FY2021-FY2025  layers available 6/6  overall DETERIORATING        gaps none


COST  FY2022-FY2026  layers available 6/6  overall STRONGLY_IMPROVING   gaps none


MSFT  FY2022-FY2026  layers available 6/6  overall DETERIORATING        gaps none
CRM   FMP FmpPlanRestrictionError: FMP income-statement for CRM is not available on the current subscription (HTTP 402): Prem
NOW   FMP FmpPlanRestrictionError: FMP income-statement for NOW is not available on the current subscription (HTTP 402): Prem


CRM and NOW are not available on the free FMP plan; the client surfaces this as a typed `FmpPlanRestrictionError` (HTTP 402) instead of a vague failure. The plan was deliberately not upgraded.

Provider differences, such as FMP supplying Visa diluted shares where SEC Company Facts do not, are intentionally left visible here. Slice 5C reconciles SEC and FMP; Slice 5B only proves that both feed the same canonical boundary.